In [42]:
import pandas as pd
import geopandas as gpd 

In [43]:
claves_dptos = pd.read_csv('./../datos/BD/claves_dptos_ref.csv')
claves_dptos = claves_dptos.loc[~claves_dptos.codprov.isna()].astype({'codprov': 'Int64', 'coddepto' : 'Int64', 'IN1' : 'Int64'})
claves_dptos['codprov'] = claves_dptos['codprov'].astype(str).str.zfill(2)
claves_dptos['coddepto'] = claves_dptos['coddepto'].astype(str).str.zfill(3)
claves_dptos['IN1'] = claves_dptos['IN1'].astype(str).str.zfill(5)
claves_dptos.loc[(claves_dptos.seccion_nombre == 'La Plata'), 'IN1'] = '06441'
claves_dptos.head()

,distrito_id,seccion_id,seccionprovincial_id,seccion_nombre,codprov,coddepto,IN1,NAM
0,1,1,0,Comuna 01,01,001,02007,Comuna 1
1,1,2,0,Comuna 02,01,002,02014,Comuna 2
2,1,3,0,Comuna 03,01,003,02021,Comuna 3
3,1,4,0,Comuna 04,01,004,02028,Comuna 4
4,1,5,0,Comuna 05,01,005,02035,Comuna 5


In [44]:
claves_dptos.loc[(claves_dptos.distrito_id == 2) & (claves_dptos.IN1.str[:2] == '60')]

,distrito_id,seccion_id,seccionprovincial_id,seccion_nombre,codprov,coddepto,IN1,NAM


In [45]:
prov_ids = claves_dptos.copy()
prov_ids['in1_prov'] = prov_ids['IN1'].astype(str).str[:2]
prov_ids = prov_ids[['distrito_id', 'in1_prov']].drop_duplicates().reset_index(drop=True)

# prov_ids = pd.read_csv('./../datos/info/radio_ref.csv')[['PROV_REF_ID', 'IDPROV']].drop_duplicates().rename(columns = {'IDPROV': 'in1_prov', 'PROV_REF_ID': 'distrito_id'}).reset_index(drop = True)
# prov_ids['in1_prov'] = prov_ids['in1_prov'].astype(str).str.zfill(2)
prov_ids.head()

,distrito_id,in1_prov
0,1,02
1,2,06
2,3,10
3,4,14
4,5,18


In [46]:
eleccion_tags = pd.read_csv('./../datos/BD/eleccion_tags.csv')
eleccion_tags.head()

,eleccion_id,año,eleccion_tipo,recuento_tipo,padron_tipo,eleccion_tag
0,1,2011,PASO,PROVISORIO,NORMAL,PASO11n
1,10,2017,PASO,PROVISORIO,NORMAL,PASO17n
2,9,2017,PASO,PROVISORIO,COMANDO,PASO17c
3,8,2017,GENERAL,PROVISORIO,NORMAL,GRAL17n
4,7,2017,GENERAL,PROVISORIO,COMANDO,GRAL17c


In [47]:
cargo = pd.read_csv('./../datos/BD/cargo_tags.csv')
cargo

,cargo_id,cargo_nombre,cargo_tag
0,3,Diputado Nacional,DN
1,1,Presidente,PR
2,4,Gobernador,GB
3,6,Diputado Provincial,DP
4,2,Senador Nacional,SN
5,5,Senador Provincial,SP
6,10,Concejal,CO
7,9,Parlamentarios Mercosur Provinciales,MP
8,8,Parlamentarios Mercosur Nacionales,MN
9,7,Intendente,IN


## Provincias

In [48]:
# provs = gpd.read_file('./../datos/info/provincia.json')[['in1', 'geometry']].rename(columns = {'in1': 'in1_prov'})
provs = gpd.read_file('./../datos/geojson/provs_simplified_100.geojson')[['in1', 'geometry']].rename(columns = {'in1': 'in1_prov'})

In [49]:

data_prov = pd.read_csv('./../datos/out/votos_agrup_prov.csv').merge(prov_ids)
# data_prov['agrupacion_nombre_'] = data_prov['agrupacion_nombre_'].fillna('NO POSITIVOS')
data_prov = data_prov.merge(eleccion_tags).merge(cargo)


In [50]:
data_prov_cargo = data_prov.copy()

# data_prov_long.groupby(['eleccion_tag', 'cargo_tag', 'in1_prov'])['agrupacion_nombre_'].agg(['unique', 'nunique'])

In [51]:
data_prov_cargo = data_prov_cargo.loc[data_prov_cargo.cargo_tag.isin(['PR', 'DN'])]
info = data_prov_cargo.groupby(['eleccion_tag', 'cargo_tag']).agg({'votos_cantidad': 'sum', 'agrupacion_nombre_': ['unique', 'nunique']})
info = info.loc[info[('votos_cantidad', 'sum')] > 10e6]
eleccion_cargo = info.sort_index().reset_index()[info.index.names]

In [52]:
eleccion_cargo.eleccion_tag.unique()

array(['BLTG15n', 'GRAL11n', 'GRAL13n', 'GRAL15n', 'GRAL17n', 'GRAL19n',
       'PASO11n', 'PASO13n', 'PASO15n', 'PASO17n', 'PASO19n', 'PASO21n',
       'PASO23n'], dtype=object)

In [53]:
eleccion_cargo.cargo_tag.unique()

array(['PR', 'DN'], dtype=object)

In [54]:
# info.sort_values(by = [('votos_cantidad', 'sum')], ascending = False)
# data_prov_cargo.agrupacion_nombre_.unique()

In [55]:
import itertools

# Define the series with two level indices and values as lists of strings
series = info['agrupacion_nombre_']['unique'].sort_index()

# # Generate the base names by combining the strings
# base_names = ['_'.join(items) for items in itertools.product(series.index.levels[0], series.values)]

# # Print the base names
# for base_name in base_names:
#     print(base_name)

In [56]:
pd.options.display.max_colwidth = 300

info['agrupacion_nombre_']['unique'].sort_index()

# eleccion_tag  cargo_tag
# BLTG15n       PR                                                                 [NO POSITIVOS, PERON, PRO]
# GRAL11n       DN                  [NO POSITIVOS, IZQUIERDA, OTROS, PERON, PRO, PROVINCIAL, SOCIALISTA, UCR]
#               PR                       [NO POSITIVOS, IZQUIERDA, OTROS, PERON, PROVINCIAL, SOCIALISTA, UCR]
# GRAL13n       DN           [NO POSITIVOS, IZQUIERDA, OTROS, PERON, PRO, MASSA, PROVINCIAL, SOCIALISTA, UCR]

eleccion_tag  cargo_tag
BLTG15n       PR                                                                                                                              [Cambiemos, Frente Para La Victoria, Resto]
GRAL11n       DN                         [Alianza Compromiso Federal, Alianza Frente Amplio Progresista, Alianza Frente Para La Victoria, Alianza Unión Para El Desarrollo Social, Resto]
              PR                         [Alianza Compromiso Federal, Alianza Frente Amplio Progresista, Alianza Frente Para La Victoria, Alianza Unión Para El Desarrollo Social, Resto]
GRAL13n       DN                                                             [Frente Para La Victoria, Resto, Frente Progresista Civico Y Social, Frente Renovador, Union Civica Radical]
GRAL15n       DN                                                   [Alianza Cambiemos, Alianza Frente Para La Victoria, Alianza Unidos Por Una Nueva Alternativa (Una), Resto, Cambiemos]
              PR           [Alianza Cambiemos,

In [57]:
# data_prov[['eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_']]


# eleccion_tag	cargo_tag	votos_tipo	agrupacion_nombre_
# 0	GRAL11n	PR	POSITIVO	Alianza Compromiso Federal
# 1	GRAL11n	PR	POSITIVO	Alianza Frente Amplio Progresista
# 2	GRAL11n	PR	POSITIVO	Alianza Frente Para La Victoria
# 3	GRAL11n	PR	POSITIVO	Alianza Unión Para El Desarrollo Social
# 4	GRAL11n	PR	BLANCO	Resto
# ...	...	...	...	...
# 3341	PASO19c	DN	COMANDO	Resto
# 3342	PASO19c	DN	COMANDO	Resto
# 3343	PASO19c	DN	COMANDO	Resto
# 3344	PASO19c	DN	COMANDO	Resto
# 3345	PASO19c	DN	COMANDO	Resto

In [58]:
# Group by 'eleccion_tag', 'cargo_tag', and 'in1_prov', and calculate the sum of 'votos_cantidad', divide for PCT
sum_votes = data_prov.groupby(['eleccion_tag', 'cargo_tag', 'in1_prov', 'votos_tipo'])['votos_cantidad'].transform('sum')
data_prov['votos_porcentaje'] = data_prov['votos_cantidad'] / sum_votes


In [59]:
# data_prov = data_prov.set_index(['in1_prov', 'eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_'])

# data_prov.index.to_frame().reset_index(drop = True).drop_duplicates()

In [60]:
data_prov = data_prov.set_index(['in1_prov', 'eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_'])


In [61]:


data_prov_table_cnt = data_prov['votos_cantidad'].unstack(['eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_'])
data_prov_table_cnt.columns = data_prov_table_cnt.columns.map('-'.join)
data_prov_table_pct = data_prov['votos_porcentaje'].unstack(['eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_'])
data_prov_table_pct.columns = data_prov_table_pct.columns.map('-'.join)

data_prov_table_cnt.to_csv('./../datos/out/votos_prov_cnt.csv')
data_prov_table_pct.to_csv('./../datos/out/votos_prov_pct.csv')

data_prov_geoms_cnt = gpd.GeoDataFrame(data_prov_table_cnt.merge(provs, left_index = True, right_on='in1_prov')).set_index('in1_prov')
data_prov_geoms_pct = gpd.GeoDataFrame(data_prov_table_pct.merge(provs, left_index = True, right_on='in1_prov')).set_index('in1_prov')

# data_prov_geoms_pct.head()

In [62]:
# data_prov_geoms_cnt.columns = [str(col).replace("(", "").replace(")", "").replace("'", "").replace(", ", "-") for col in data_prov_geoms_cnt.columns]
# data_prov_geoms_pct.columns = [str(col).replace("(", "").replace(")", "").replace("'", "").replace(", ", "-") for col in data_prov_geoms_pct.columns]

data_prov_geoms_cnt.fillna(0).to_file('./../datos/geojson/votos_cnt_prov.geojson', driver='GeoJSON') # Save the GeoDataFrame to GeoJSON
data_prov_geoms_pct.fillna(0).to_file('./../datos/geojson/votos_pct_prov.geojson', driver='GeoJSON') # Save the GeoDataFrame to GeoJSON


## Circuitos

In [63]:

# data_circ = pd.read_csv('./../datos/out/votos_circ.csv').merge(prov_ids)
data_circ = pd.read_csv('./../datos/out/votos_agrup_circ.csv').merge(prov_ids)
# data_circ['agrupacion_nombre_'] = data_circ['agrupacion_nombre_'].fillna('NO POSITIVOS')

# Guardar ref provs - dptos - circs
dist_secc_circ = data_circ.copy() # save for later

# data_circ
data_circ = data_circ.merge(eleccion_tags).merge(cargo)


# Group by 'eleccion_tag', 'cargo_tag', and 'in1_prov', and calculate the sum of 'votos_cantidad', divide for PCT
sum_votes = data_circ.groupby(['eleccion_tag', 'cargo_tag', 'in1_prov', 'votos_tipo', 'circuito_id'])['votos_cantidad'].transform('sum')
data_circ['votos_porcentaje'] = data_circ['votos_cantidad'] / sum_votes


data_circ = data_circ.set_index(['distrito_id', 'circuito_id', 'eleccion_tag', 'cargo_tag', 'agrupacion_nombre_', 'votos_tipo'])


data_circ_table_cnt = data_circ['votos_cantidad'].unstack(['eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_'])
data_circ_table_pct = data_circ['votos_porcentaje'].unstack(['eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_'])

data_circ_table_cnt.to_csv('./../datos/out/votos_circ_cnt.csv')
data_circ_table_pct.to_csv('./../datos/out/votos_circ_pct.csv')


In [64]:
info_intervals = data_circ_table_pct.copy()

quantile_01 = info_intervals.quantile(0.1)
quantile_09 = info_intervals.quantile(0.85)

quantiles = pd.concat([quantile_01, quantile_09], axis = 1)
magnitudes = quantiles.reset_index().groupby('agrupacion_nombre_')[[.1, .85]].mean()
magnitudes.to_csv('./../web/agrup_nombre_magnitudes.csv')

In [65]:
# circuitos = gpd.read_file('/home/matias/repos/CNE-INDEC-georef/mapaelectoral/circuitos-CNE-TTGL.geojson')
# circuitos = gpd.read_file('./../datos/info/circuitos_dpto_prov.geojson').merge(prov_ids)
# circuitos = gpd.read_file('./../datos/geojson/circs_simplified_1.geojson').merge(prov_ids)
circuitos = gpd.read_file('./../datos/geojson/circs_ref_simplified_1.geojson').merge(prov_ids)
circuitos.loc[(circuitos.seccion_nombre == 'La Plata'), 'IN1'] = '06441'
circuitos['circuito_id'] = circuitos['circuito'].str.zfill(6)
circuitos['in1_prov'] = circuitos['IN1'].str[0:2]


CNE_in1 = circuitos[['distrito_id', 'circuito_id', 'in1_prov']]
dist_secc_in1 = dist_secc_circ.merge(CNE_in1, on = ['distrito_id', 'circuito_id', 'in1_prov'])
dist_secc_in1 = dist_secc_in1.groupby(['distrito_id', 'seccion_id', 'in1_prov'])['votos_cantidad'].sum().reset_index()

circuitos = circuitos[['distrito_id', 'in1_prov', 'circuito_id', 'geometry']]
circuitos.tail()

,distrito_id,in1_prov,circuito_id,geometry
5455,15.0,58,000440,"POLYGON Z ((-69.68947 -37.10574 0.00000, -69.68931 -37.10692 0.00000, -69.68952 -37.10836 0.00000, -69.68914 -37.10943 0.00000, -69.68802 -37.11033 0.00000, -69.68364 -37.11278 0.00000, -69.68052 -37.11416 0.00000, -69.67896 -37.11449 0.00000, -69.67479 -37.11499 0.00000, -69.67332 -37.11548 0.0..."
5456,15.0,58,000450,"POLYGON Z ((-70.05038 -36.96967 0.00000, -70.04261 -36.66099 0.00000, -70.04138 -36.66111 0.00000, -70.03944 -36.66249 0.00000, -70.03812 -36.66307 0.00000, -70.03643 -36.66280 0.00000, -70.03483 -36.66317 0.00000, -70.03404 -36.66311 0.00000, -70.03182 -36.66349 0.00000, -70.02924 -36.66336 0.0..."
5457,15.0,58,000690,"POLYGON Z ((-70.92760 -36.47619 0.00000, -70.92152 -36.47497 0.00000, -70.91634 -36.47446 0.00000, -70.91401 -36.47545 0.00000, -70.91180 -36.47573 0.00000, -70.91017 -36.47551 0.00000, -70.90841 -36.47614 0.00000, -70.90694 -36.47640 0.00000, -70.90546 -36.47583 0.00000, -70.90369 -36.47598 0.0..."
5458,15.0,58,001650,"POLYGON Z ((-71.15259 -41.05513 0.00000, -71.15290 -41.05425 0.00000, -71.15277 -41.05393 0.00000, -71.15319 -41.05325 0.00000, -71.15403 -41.05307 0.00000, -71.15459 -41.05272 0.00000, -71.15655 -41.05097 0.00000, -71.15828 -41.05013 0.00000, -71.15865 -41.04988 0.00000, -71.15889 -41.04940 0.0..."
5459,15.0,58,001360,"POLYGON Z ((-70.46011 -40.24839 0.00000, -70.37625 -40.18850 0.00000, -70.17827 -40.12011 0.00000, -69.99345 -40.19262 0.00000, -70.00209 -40.20320 0.00000, -70.00598 -40.20674 0.00000, -70.00794 -40.20763 0.00000, -70.01012 -40.20808 0.00000, -70.01915 -40.20913 0.00000, -70.02171 -40.20983 0.0..."


In [66]:

data_circ_table_cnt.columns = data_circ_table_cnt.columns.map('-'.join)
data_circ_geoms_cnt = gpd.GeoDataFrame(data_circ_table_cnt.merge(circuitos, left_index = True, right_on=['distrito_id', 'circuito_id'])).set_index(['distrito_id', 'circuito_id'])

data_circ_table_pct.columns = data_circ_table_pct.columns.map('-'.join)
data_circ_geoms_pct = gpd.GeoDataFrame(data_circ_table_pct.merge(circuitos, left_index = True, right_on=['distrito_id', 'circuito_id'])).set_index(['distrito_id', 'circuito_id'])

data_circ_geoms_cnt.columns = [str(col).replace("(", "").replace(")", "").replace("'", "").replace(", ", "-") for col in data_circ_geoms_cnt.columns]
data_circ_geoms_pct.columns = [str(col).replace("(", "").replace(")", "").replace("'", "").replace(", ", "-") for col in data_circ_geoms_pct.columns]

# data_circ_geoms_cnt.head(1000).to_file('./../datos/geojson/votos_cnt_circ_lite.geojson', driver='GeoJSON') # Save the GeoDataFrame to GeoJSON
# data_circ_geoms_pct.head(1000).to_file('./../datos/geojson/votos_pct_circ_lite.geojson', driver='GeoJSON') # Save the GeoDataFrame to GeoJSON

data_circ_geoms_cnt.to_file('./../datos/geojson/votos_cnt_circ.geojson', driver='GeoJSON') # Save the GeoDataFrame to GeoJSON
data_circ_geoms_pct.to_file('./../datos/geojson/votos_pct_circ.geojson', driver='GeoJSON') # Save the GeoDataFrame to GeoJSON

data_prov_geoms_pct.head()

,GRAL11n-PR-POSITIVO-Alianza Compromiso Federal,GRAL11n-PR-POSITIVO-Alianza Frente Amplio Progresista,GRAL11n-PR-POSITIVO-Alianza Frente Para La Victoria,GRAL11n-PR-POSITIVO-Alianza Unión Para El Desarrollo Social,GRAL11n-PR-BLANCO-Resto,GRAL11n-PR-IMPUGNADO-Resto,GRAL11n-PR-NULO-Resto,GRAL11n-PR-POSITIVO-Resto,GRAL11n-PR-RECURRIDO-Resto,PASO11n-PR-POSITIVO-Alianza Frente Amplio Progresista,...,PASO23n-DN-BLANCO-Resto,PASO23n-DN-COMANDO-Resto,PASO23n-DN-IMPUGNADO-Resto,PASO23n-DN-NULO-Resto,PASO23n-DN-POSITIVO-Resto,PASO23n-DN-RECURRIDO-Resto,PASO23n-DN-POSITIVO-Union Por La Patria,GRAL19c-DN-COMANDO-Resto,PASO19c-DN-COMANDO-Resto,geometry
in1_prov,,,,,,,,,,,,,,,,,,,,,
02,0.098744,0.277848,0.350492,0.094405,1.0,1.0,1.0,0.178510,1.0,0.143947,...,1.0,NaN,1.0,1.0,0.275197,1.0,NaN,NaN,NaN,"POLYGON ((-58.45535 -34.52776, -58.36164 -34.58116, -58.36792 -34.59754, -58.34685 -34.60236, -58.33515 -34.62707, -58.37405 -34.65719, -58.42379 -34.66225, -58.46121 -34.70541, -58.52888 -34.65463, -58.53151 -34.61551, -58.50077 -34.54946, -58.45535 -34.52776))"
06,0.073554,0.149715,0.562835,0.097022,1.0,1.0,1.0,0.116874,1.0,0.081853,...,1.0,1.0,1.0,1.0,0.045684,1.0,0.349203,1.0,1.0,"MULTIPOLYGON (((-62.15339 -40.45148, -62.15854 -40.45646, -62.17670 -40.45436, -62.15783 -40.44676, -62.15339 -40.45148)), ((-62.18040 -40.46090, -62.15662 -40.45768, -62.21714 -40.52348, -62.23491 -40.50795, -62.20094 -40.44664, -62.18040 -40.46090)), ((-62.13386 -40.41127, -62.13602 -40.40189,..."
10,0.017860,0.051087,0.697690,0.179234,1.0,1.0,1.0,0.054129,1.0,0.023774,...,1.0,1.0,1.0,1.0,0.024828,1.0,0.535571,1.0,1.0,"POLYGON ((-68.50537 -25.16851, -67.86460 -25.23840, -66.87927 -25.24173, -66.55819 -25.30065, -66.59980 -25.36225, -66.53710 -25.37248, -66.51855 -25.44033, -66.53525 -25.46226, -66.53456 -25.53651, -66.57277 -25.58672, -66.56390 -25.62176, -66.60646 -25.62867, -66.64858 -25.71203, -66.80173 -25..."
14,0.122967,0.234096,0.373439,0.179725,1.0,1.0,1.0,0.089773,1.0,0.145240,...,1.0,1.0,1.0,1.0,0.298790,1.0,0.082436,NaN,1.0,"POLYGON ((-63.89431 -29.62860, -63.87386 -29.63079, -63.84191 -29.58077, -63.78377 -29.58417, -63.79605 -29.61443, -63.75615 -29.62155, -63.74841 -29.65328, -63.71900 -29.66622, -63.62790 -29.65500, -63.46535 -29.66448, -63.48047 -29.72555, -63.45970 -29.72552, -63.45962 -29.75693, -62.80634 -29..."
18,0.040311,0.071972,0.679974,0.128933,1.0,1.0,1.0,0.078809,1.0,0.032679,...,1.0,1.0,1.0,1.0,0.405679,1.0,0.306975,1.0,1.0,"MULTIPOLYGON (((-56.86162 -27.59213, -56.85520 -27.59290, -56.85571 -27.59483, -56.86625 -27.59188, -56.86162 -27.59213)), ((-56.75908 -27.59105, -56.75339 -27.59210, -56.74942 -27.59346, -56.74911 -27.59396, -56.75908 -27.59105)), ((-56.87428 -27.58250, -56.86358 -27.58762, -56.86224 -27.58876,..."


In [67]:
## DEBUG de merge geometria circuitos

# ## Por problemas en merge de circuito, lo que se mantiene es el 96.66% de los votos total
# 100*data_circ_geoms.dropna().votos_cantidad.sum()/data_circ_geoms.votos_cantidad.sum()

## Verificar los casos con mas votos perdidos por el merge
# data_circ_geoms.loc[data_circ_geoms.geometry.isna()].groupby('distrito_id')['votos_cantidad'].sum().sort_values()
# data_circ_geoms.loc[data_circ_geoms.geometry.isna()].groupby(['distrito_id', 'circuito_id'])['votos_cantidad'].sum().sort_values().tail(10)

# ## Excepto por 4 casos puntuales, se puede decir que el id de circuito es unico en cada provincia. Es decir se puede unir por provincia y circuito.
# circuitos.groupby(['in1_prov', 'circuito']).in1_dpto.nunique().sort_values(ascending=False).head()
# # in1_prov  circuito
# # 14        sindatos    3
# #           zonagris    3
# # 22        00072       2
# # 30        00303       2
# # 58        01550       1

### Referencia seccion_id - in1_dpto

Se obtiene tomando para cada seccion_id el in1_dpto que mas votos acumula. Se asume que el in1_dpto que mas se repite es el correcto.


In [68]:

# # take n largest
# # largest_intersections = intersections.groupby(['codprov', 'coddepto', 'circuito']).apply(lambda x: x.nlargest(1, 'area')).reset_index(drop=True)

# largest = dist_secc_in1.groupby(['distrito_id', 'seccion_id']).apply(lambda x: x.nlargest(1, 'votos_cantidad')).reset_index(drop=True)

# largest[['distrito_id', 'seccion_id', 'in1_prov', 'in1_dpto']].to_csv('./../datos/info/seccion_dpto.csv', index = False)

## Departamentos (Secciones)

In [69]:
dptos = gpd.read_file('./../datos/geojson/dptos_simplified_2.geojson')[['in1', 'geometry']].rename(columns = {'in1': 'in1_dpto'})
dptos['in1_dpto'] = dptos['in1_dpto'].astype(int)
# dptos['in1_prov'] = dptos['in1_dpto'].astype(str).str[:2].astype(int)
dptos.head()

,in1_dpto,geometry
0,30015,"POLYGON ((-58.38761 -30.82889, -58.38711 -30.83035, -58.38629 -30.83156, -58.38421 -30.83035, -58.38427 -30.82917, -58.38383 -30.82897, -58.38237 -30.82911, -58.38067 -30.83054, -58.37921 -30.83056, -58.37771 -30.83004, -58.37708 -30.83060, -58.37743 -30.83221, -58.37707 -30.83274, -58.37544 -30..."
1,30035,"POLYGON ((-58.96265 -30.60344, -58.96104 -30.60525, -58.95838 -30.60571, -58.95797 -30.60663, -58.95701 -30.60748, -58.95574 -30.60744, -58.95448 -30.60633, -58.95368 -30.60691, -58.95383 -30.60828, -58.95354 -30.60917, -58.95210 -30.61026, -58.95083 -30.61028, -58.95031 -30.60997, -58.95001 -30..."
2,30056,"POLYGON ((-59.06547 -32.48629, -59.06444 -32.48675, -59.06367 -32.48661, -59.06291 -32.48582, -59.06142 -32.48542, -59.05723 -32.48683, -59.05614 -32.48804, -59.05484 -32.48873, -59.05324 -32.48898, -59.05211 -32.48845, -59.05113 -32.48897, -59.05031 -32.48835, -59.04674 -32.49037, -59.04662 -32..."
3,30063,"POLYGON ((-58.43841 -33.51761, -58.44346 -33.53784, -58.44408 -33.53964, -58.44501 -33.54105, -58.44769 -33.54360, -58.47775 -33.56409, -58.48004 -33.56597, -58.49376 -33.58146, -58.49467 -33.58369, -58.49464 -33.58719, -58.48493 -33.61455, -58.47674 -33.64627, -58.47535 -33.65024, -58.47349 -33..."
4,30105,"POLYGON ((-60.16654 -32.27211, -60.16239 -32.26576, -60.14369 -32.27519, -60.13687 -32.26628, -60.13256 -32.26812, -60.13123 -32.26897, -60.12970 -32.27041, -60.11953 -32.28455, -60.11886 -32.28691, -60.11892 -32.28916, -60.11988 -32.29655, -60.11840 -32.30029, -60.11586 -32.30344, -60.09799 -32..."


In [70]:
seccion_dpto = pd.read_csv('./../datos/info/seccion_dpto.csv')
seccion_dpto.head()

,distrito_id,seccion_id,in1_prov,in1_dpto
0,1,1,2,2007
1,1,2,2,2014
2,1,3,2,2021
3,1,4,2,2028
4,1,5,2,2035


In [71]:
seccion_dpto.merge(dptos).head()

,distrito_id,seccion_id,in1_prov,in1_dpto,geometry
0,1,1,2,2007,"POLYGON ((-58.38609 -34.57823, -58.38320 -34.57833, -58.37980 -34.58018, -58.37613 -34.57997, -58.37509 -34.58054, -58.37387 -34.57871, -58.36789 -34.57823, -58.36532 -34.57963, -58.36525 -34.58016, -58.37063 -34.58059, -58.37062 -34.58082, -58.36904 -34.58174, -58.36164 -34.58116, -58.36142 -34..."
1,1,2,2,2014,"POLYGON ((-58.38359 -34.57339, -58.38180 -34.57440, -58.37552 -34.57392, -58.37077 -34.57662, -58.37076 -34.57708, -58.37610 -34.57752, -58.37434 -34.57871, -58.37387 -34.57871, -58.37509 -34.58054, -58.37613 -34.57997, -58.37980 -34.58018, -58.38354 -34.57825, -58.38983 -34.57822, -58.39161 -34..."
2,1,3,2,2021,"POLYGON ((-58.41026 -34.59811, -58.40447 -34.59805, -58.40198 -34.59939, -58.39869 -34.59978, -58.39290 -34.59966, -58.39235 -34.60317, -58.39177 -34.61130, -58.39114 -34.62727, -58.40833 -34.62953, -58.41175 -34.63037, -58.41230 -34.62305, -58.41201 -34.62299, -58.41240 -34.62190, -58.41284 -34..."
3,1,4,2,2028,"POLYGON ((-58.35517 -34.61945, -58.35404 -34.61972, -58.35349 -34.61942, -58.35266 -34.61979, -58.35253 -34.61956, -58.35189 -34.61964, -58.35158 -34.61925, -58.34941 -34.61975, -58.34774 -34.61896, -58.34649 -34.61911, -58.34526 -34.61875, -58.34461 -34.61896, -58.34135 -34.61843, -58.33837 -34..."
4,1,5,2,2035,"POLYGON ((-58.42271 -34.59773, -58.41189 -34.59802, -58.41387 -34.60726, -58.41407 -34.60793, -58.41463 -34.60821, -58.41443 -34.61075, -58.41284 -34.61413, -58.41149 -34.63287, -58.41045 -34.63525, -58.41123 -34.63812, -58.42346 -34.64022, -58.42577 -34.63130, -58.42922 -34.61512, -58.42999 -34..."


In [72]:

# data_circ = data_circ.merge(eleccion_tags).set_index(['in1_prov', 'circuito_id', 'eleccion_tag', 'cargo_tag', 'agrupacion_nombre_'])
# data_circ_table = data_circ['votos_cantidad'].unstack(['eleccion_tag', 'cargo_tag', 'agrupacion_nombre_'])

# data_circ_table

In [73]:
data_secc = pd.read_csv('./../datos/out/votos_agrup_secc.csv').merge(seccion_dpto.merge(dptos), how = 'left') ## Solo se pierde la seccion 24-3 que es la Antartida Argentina
data_secc['in1_prov'] = data_secc['in1_prov'].fillna(24).astype(int)#.astype(str).str.zfill(2)
data_secc['in1_dpto'] = data_secc['in1_dpto'].fillna(24003).astype(int)#.astype(str).str.zfill(5)
data_secc['agrupacion_nombre_'] = data_secc['agrupacion_nombre_'].fillna('NO POSITIVOS')

# .set_index(['in1_prov', 'circuito_id', 'eleccion_tag', 'cargo_tag', 'agrupacion_nombre_'])
# # data_circ_table = data_circ['votos_cantidad'].unstack(['eleccion_tag', 'cargo_tag', 'agrupacion_nombre_'])

data_secc = data_secc.merge(eleccion_tags).merge(cargo)

# data_secc = data_secc.set_index(['in1_prov', 'in1_dpto', 'eleccion_tag', 'cargo_tag', 'agrupacion_nombre_'])
# data_secc_table = data_secc['votos_cantidad'].unstack(['eleccion_tag', 'cargo_tag', 'agrupacion_nombre_'])

## Replace by the following, as there are a few which are duplicate for some reason..
data_secc = data_secc.groupby(['in1_prov', 'in1_dpto', 'eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_'])['votos_cantidad'].sum().reset_index()

# grouped = data_secc[data_secc['votos_tipo'].isin(['POSITIVO', 'BLANCO'])].groupby(['in1_prov', 'in1_dpto', 'eleccion_tag', 'cargo_tag', 'agrupacion_nombre_'])
# sum_votes_positivo_blanco = grouped['votos_cantidad'].sum().reset_index()


In [74]:
data_secc = data_secc.groupby(['in1_prov', 'in1_dpto', 'eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_'])['votos_cantidad'].sum().reset_index()
data_secc['votos_porcentaje'] = data_secc['votos_cantidad'] / sum_votes


In [75]:

# Group by 'eleccion_tag', 'cargo_tag', and 'in1_prov', and calculate the sum of 'votos_cantidad', divide for PCT
sum_votes = data_secc.groupby(['eleccion_tag', 'cargo_tag', 'votos_tipo', 'in1_prov', 'in1_dpto'])['votos_cantidad'].transform('sum')
data_secc['votos_porcentaje'] = data_secc['votos_cantidad'] / sum_votes
data_secc_pcts = data_secc.copy()

## Guardar para sabes culaes son los valores tipicos y de ahi definir escalas de colores.
data_secc_pcts.to_csv('./../datos/info/data_secc_pct_vals.csv', index = False)

data_secc = data_secc.set_index(['in1_prov', 'in1_dpto', 'eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_'])


data_secc_table_cnt = data_secc['votos_cantidad'].unstack(['eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_'])
data_secc_table_pct = data_secc['votos_porcentaje'].unstack(['eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_'])


In [76]:
data_secc_table_cnt.columns = data_secc_table_cnt.columns.map('-'.join)
data_secc_table_pct.columns = data_secc_table_pct.columns.map('-'.join)


In [77]:
data_secc_table_pct.columns

Index(['BLTG15n-PR-BLANCO-Resto', 'BLTG15n-PR-IMPUGNADO-Resto',
       'BLTG15n-PR-NULO-Resto', 'BLTG15n-PR-POSITIVO-Cambiemos',
       'BLTG15n-PR-POSITIVO-Frente Para La Victoria',
       'BLTG15n-PR-RECURRIDO-Resto', 'GRAL11n-DN-BLANCO-Resto',
       'GRAL11n-DN-IMPUGNADO-Resto', 'GRAL11n-DN-NULO-Resto',
       'GRAL11n-DN-POSITIVO-Alianza Compromiso Federal',
       ...
       'PASO17n-DN-POSITIVO-Cambiemos',
       'PASO17n-DN-POSITIVO-Frente Justicialista',
       'PASO17n-DN-POSITIVO-Unidad Ciudadana', 'PASO19c-DN-COMANDO-Resto',
       'PASO19c-PR-COMANDO-Resto', 'PASO23n-DN-POSITIVO-Union Por La Patria',
       'GRAL13n-DN-POSITIVO-Union Civica Radical',
       'PASO13n-DN-POSITIVO-Union Civica Radical',
       'PASO15n-DN-POSITIVO-Alianza Union Por Cordoba',
       'PASO21n-DN-POSITIVO-Hacemos Por Córdoba'],
      dtype='object', length=186)

### Check pct values por fuerza

In [78]:

data_secc_geoms_cnt = gpd.GeoDataFrame(data_secc_table_cnt.merge(seccion_dpto.merge(dptos), left_index = True, right_on=['in1_prov', 'in1_dpto'])).set_index(['distrito_id', 'seccion_id'])
data_secc_geoms_pct = gpd.GeoDataFrame(data_secc_table_pct.merge(seccion_dpto.merge(dptos), left_index = True, right_on=['in1_prov', 'in1_dpto'])).set_index(['distrito_id', 'seccion_id'])

data_secc_geoms_cnt.columns = [str(col).replace("(", "").replace(")", "").replace("'", "").replace(", ", "-") for col in data_secc_geoms_cnt.columns]
data_secc_geoms_pct.columns = [str(col).replace("(", "").replace(")", "").replace("'", "").replace(", ", "-") for col in data_secc_geoms_pct.columns]

data_secc_geoms_cnt.to_file('./../datos/geojson/votos_cnt_secc.geojson', driver = 'GeoJSON')
data_secc_geoms_pct.to_file('./../datos/geojson/votos_pct_secc.geojson', driver = 'GeoJSON') # Save the GeoDataFrame to GeoJSON

# data_prov_geoms_pct.head()

## Upload to Mapbox as a tileset

In [79]:
import os
os.environ['MAPBOX_ACCESS_TOKEN'] = 'sk.eyJ1IjoibWF0dXRlaWdsZXNpYXMyIiwiYSI6ImNrb3lvMWZyajAxZncycG8ycnJkaTI1ZjYifQ.LXJGImmBgQtWWrNOC1wTcA'


In [80]:
# # !cd /home/matias/Documents/electoral/datos/geojson
# # !export MAPBOX_ACCESS_TOKEN=sk.eyJ1IjoibWF0dXRlaWdsZXNpYXMyIiwiYSI6ImNrb3lvMWZyajAxZncycG8ycnJkaTI1ZjYifQ.LXJGImmBgQtWWrNOC1wTcA

# !tilesets delete-source -f matuteiglesias2 votos_provs_pct_3
# !tilesets delete-source -f matuteiglesias2 votos_dptos_pct_3
# !tilesets delete-source -f matuteiglesias2 votos_circs_pct_3

# !tilesets delete -f matuteiglesias2.votos_pct_3

In [81]:
# pip install mapbox-tilesets

In [82]:
## HOGARES. 
# (REPEAT FOR PERSONAS, M24, ETC.)
!tilesets upload-source matuteiglesias2 votos_provs_pct_5 ./../datos/geojson/votos_pct_prov.geojson
!tilesets upload-source matuteiglesias2 votos_dptos_pct_5 ./../datos/geojson/votos_pct_secc.geojson
# !tilesets upload-source matuteiglesias2 votos_circs_pct ./../datos/geojson/votos_pct_circ_lite.geojson
!tilesets upload-source matuteiglesias2 votos_circs_pct_5 ./../datos/geojson/votos_pct_circ.geojson


# # tilesets create username.hello-world-tiles --recipe hello-world-recipe.json --name "hello world"
!tilesets create matuteiglesias2.votos_pct_5 --recipe ./../datos/geojson/votos_pct-recipe-pct5.json --name "Votos - Porcentaje"
!tilesets publish matuteiglesias2.votos_pct_5


upload progress  [-------------------------------------------------------]    0%
{"id": "mapbox://tileset-source/matuteiglesias2/votos_provs_pct_5", "files": 1, "source_size": 1043319, "file_size": 1043319}
upload progress  [-------------------------------------------------------]    0%
{"id": "mapbox://tileset-source/matuteiglesias2/votos_dptos_pct_5", "files": 1, "source_size": 17233144, "file_size": 17233144}
upload progress  [-------------------------------------------------------]    0%
{"id": "mapbox://tileset-source/matuteiglesias2/votos_circs_pct_5", "files": 1, "source_size": 64877040, "file_size": 64877040}
{"message": "Successfully created empty tileset matuteiglesias2.votos_pct_5. Publish your tileset to begin processing your data into vector tiles."}
{"message": "Processing matuteiglesias2.votos_pct_5", "jobId": "cllb9zk2f000s08kz6o6q4f4a"}

✔ Tileset job received. Visit https://studio.mapbox.com/tilesets/matuteiglesias2.votos_pct_5 or run tilesets job matuteiglesias2.voto

In [83]:
xx

NameError: name 'xx' is not defined

In [ ]:
import os
os.environ['MAPBOX_ACCESS_TOKEN'] = 'sk.eyJ1IjoibWF0dXRlaWdsZXNpYXMyIiwiYSI6ImNrb3lvMWZyajAxZncycG8ycnJkaTI1ZjYifQ.LXJGImmBgQtWWrNOC1wTcA'


In [ ]:
# !cd /home/matias/Documents/electoral/datos/geojson
# !export MAPBOX_ACCESS_TOKEN=sk.eyJ1IjoibWF0dXRlaWdsZXNpYXMyIiwiYSI6ImNrb3lvMWZyajAxZncycG8ycnJkaTI1ZjYifQ.LXJGImmBgQtWWrNOC1wTcA

!tilesets delete-source -f matuteiglesias2 votos_provs
!tilesets delete-source -f matuteiglesias2 votos_dptos
!tilesets delete-source -f matuteiglesias2 votos_circs

!tilesets delete -f matuteiglesias2.votos

Source deleted.
Source deleted.
Source deleted.
Tileset deleted.
Traceback (most recent call last):
  File "/home/matias/anaconda3/lib/python3.9/site-packages/cligj/features.py", line 192, in coords_from_query
    coords = json.loads(query)
  File "/home/matias/anaconda3/lib/python3.9/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
  File "/home/matias/anaconda3/lib/python3.9/json/decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
  File "/home/matias/anaconda3/lib/python3.9/json/decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/matias/anaconda3/bin/tilesets", line 8, in <module>
    sys.exit(cli())
  File "/home/matias/anaconda3/lib/python3.9/site-packages/click/core.py", line 11

In [ ]:
## HOGARES. 
# (REPEAT FOR PERSONAS, M24, ETC.)
!tilesets upload-source matuteiglesias2 votos_provs ./../datos/geojson/votos_prov.geojson
!tilesets upload-source matuteiglesias2 votos_dptos ./../datos/geojson/votos_secc.geojson
!tilesets upload-source matuteiglesias2 votos_circs ./../datos/geojson/votos_circ_lite.geojson


# # tilesets create username.hello-world-tiles --recipe hello-world-recipe.json --name "hello world"
!tilesets create matuteiglesias2.votos --recipe ./../datos/geojson/votos-recipe.json --name "Votos - Cantidad"
!tilesets publish matuteiglesias2.votos

upload progress  [-------------------------------------------------------]    0%
{"id": "mapbox://tileset-source/matuteiglesias2/votos_dptos", "files": 1, "source_size": 3824021, "file_size": 3824021}
upload progress  [-------------------------------------------------------]    0%
{"id": "mapbox://tileset-source/matuteiglesias2/votos_circs", "files": 1, "source_size": 1800597, "file_size": 1800597}
{"message": "Successfully created empty tileset matuteiglesias2.votos. Publish your tileset to begin processing your data into vector tiles."}
{"message": "Processing matuteiglesias2.votos", "jobId": "clhv5ddrx001f08laa4m60ts7"}

✔ Tileset job received. Visit https://studio.mapbox.com/tilesets/matuteiglesias2.votos or run tilesets job matuteiglesias2.votos clhv5ddrx001f08laa4m60ts7 to view the status of your tileset.
